In [4]:
import pandas as pd        # tables (Class 9)
import numpy as np         # numerical operations
import archetypes as arch  # AA and ADA, the method from the paper
import sklearn             # scaler, PCA, K-Means, KNN (Class 10)
import plotly              # interactive chartsz

print("pandas", pd.__version__)
print("archetypes", arch.AA)
print("sklearn", sklearn.__version__)
print("plotly", plotly.__version__)

pandas 3.1.1
archetypes <class 'archetypes.numpy._aa.AA'>
sklearn 1.9.1
plotly 7.1.0


# 1. Data collection
Player and team statistics from the Liga Femenina Endesa, from the official stats site
of the Spanish Basketball Federation (baloncestoenvivo.feb.es).
We test on one page first, then run it for the whole league.

In [5]:
import requests            # downloads web pages
from io import StringIO    # wraps text so pandas can read it like a file

In [7]:
# Official source: Spanish Basketball Federation (FEB) stats site
url = "https://baloncestoenvivo.feb.es/estadisticas.aspx?g=4&t=2026&nm=lfendesa"

headers = {"User-Agent": "Mozilla/5.0"}   # identify ourselves as a normal browser

response = requests.get(url, headers=headers)

print(response.status_code)   # 200 = OK, 403 = blocked, 404 = page not found

200


In [9]:
tables = pd.read_html(StringIO(response.text))   # every <table> in the page becomes a DataFrame

print("Tables found:", len(tables))

for i, t in enumerate(tables):   # position + table, one per loop
    print(i, t.shape)            # (rows, columns)

Tables found: 1
0 (16, 20)


In [10]:
df_teams = tables[0]   # the only table on the page: one row per team
df_teams.head()

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
                     Equipo               Part                MIN   
0             MEINS AVENIDA                  7           1400 200   
1               CADI LA SEU                  7           1400 200   
2           MIRAI EUSKOTREN                  7           1425 203   
3  INVERSUS GERNIKA BIZKAIA                  7           1450 207   
4       CASADEMONT ZARAGOZA                  7           1400 200   

  Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Unnamed: 6_level_0  \
                  PT                 T2                 T3                 TC   
0          500  71,4       127/276  46%      57/156  36,5%     184/432  42,6%   
1          449  64,1     125/275  45,5%      40/137  29,2%       165/412  40%   
2          491  70,1     146/320  45,6%        36/109  33%     182/429  42,4%   
3          541  77,3     155/296  52,4%        42/150  28%     197/446  44,2%   
4            581  83     139/263  52,9%      72/196  36,7%       211/459  46%   

  Unnamed: 7_level_0   Rebotes                       Unnamed: 11_level_0  \
                  TL        RO         RD         RT                  AS   
0      75/106  70,8%     56  8  141  20,1  197  28,1           106  15,1   
1      79/109  72,5%   58  8,3  185  26,4  243  34,7            86  12,3   
2      91/121  75,2%  71  10,1  146  20,9    217  31            90  12,9   
3     105/153  68,6%   65  9,3    168  24  233  33,3           103  14,7   
4      87/120  72,5%  88  12,6  208  29,7  296  42,3           138  19,7   

  Unnamed: 12_level_0 Unnamed: 13_level_0  Tapones           \
                   BR                  BP       TF       TC   
0              70  10            99  14,1  10  1,4  16  2,3   
1             67  9,6           102  14,6  29  4,1   8  1,1   
2             67  9,6           103  14,7  17  2,4  12  1,7   
3             62  8,9             105  15  11  1,6  10  1,4   
4             60  8,6           106  15,1   9  1,3   9  1,3   

  Unnamed: 16_level_0     Faltas            Unnamed: 19_level_0  
                   MT         FC         FR                  VA  
0                0  0  132  18,9  125  17,9           498  71,1  
1              2  0,3  115  16,4  116  16,6           496  70,9  
2                0  0    126  18  135  19,3             511  73  
3                0  0  124  17,7  136  19,4             560  80  
4                0  0  127  18,1  138  19,7          708  101,1

In [11]:
print(df_teams.columns.tolist())   # column names as a plain list
df_teams.dtypes                    # data type of each column

[('Unnamed: 0_level_0', 'Equipo'), ('Unnamed: 1_level_0', 'Part'), ('Unnamed: 2_level_0', 'MIN'), ('Unnamed: 3_level_0', 'PT'), ('Unnamed: 4_level_0', 'T2'), ('Unnamed: 5_level_0', 'T3'), ('Unnamed: 6_level_0', 'TC'), ('Unnamed: 7_level_0', 'TL'), ('Rebotes', 'RO'), ('Rebotes', 'RD'), ('Rebotes', 'RT'), ('Unnamed: 11_level_0', 'AS'), ('Unnamed: 12_level_0', 'BR'), ('Unnamed: 13_level_0', 'BP'), ('Tapones', 'TF'), ('Tapones', 'TC'), ('Unnamed: 16_level_0', 'MT'), ('Faltas', 'FC'), ('Faltas', 'FR'), ('Unnamed: 19_level_0', 'VA')]


Unnamed: 0_level_0   Equipo      str
Unnamed: 1_level_0   Part      int64
Unnamed: 2_level_0   MIN         str
Unnamed: 3_level_0   PT          str
Unnamed: 4_level_0   T2          str
Unnamed: 5_level_0   T3          str
Unnamed: 6_level_0   TC          str
Unnamed: 7_level_0   TL          str
Rebotes              RO          str
                     RD          str
                     RT          str
Unnamed: 11_level_0  AS          str
Unnamed: 12_level_0  BR          str
Unnamed: 13_level_0  BP          str
Tapones              TF          str
                     TC          str
Unnamed: 16_level_0  MT          str
Faltas               FC          str
                     FR          str
Unnamed: 19_level_0  VA          str
dtype: object

In [12]:
# Original (Spanish) headers -> clean English names, in the same order as the table
clean_columns = [
    "team",         # Equipo
    "games",        # Part   = partidos jugados
    "min",          # MIN
    "pts",          # PT     = puntos
    "fg2",          # T2     = tiros de 2 (made/attempted %)
    "fg3",          # T3     = tiros de 3
    "fg",           # TC     = tiros de campo (2s + 3s)
    "ft",           # TL     = tiros libres
    "oreb",         # RO     = rebotes ofensivos
    "dreb",         # RD     = rebotes defensivos
    "reb",          # RT     = rebotes totales
    "ast",          # AS     = asistencias
    "stl",          # BR     = balones recuperados
    "tov",          # BP     = balones perdidos
    "blk",          # TF     = tapones a favor
    "blk_against",  # TC     = tapones en contra (shots blocked by the opponent)
    "mt",           # MT     = not defined on the page, we check it later
    "pf",           # FC     = faltas cometidas
    "fd",           # FR     = faltas recibidas
    "val",          # VA     = valoración (efficiency rating)
]

print(len(clean_columns), "names for", df_teams.shape[1], "columns")   # must match: 20 and 20

df_teams.columns = clean_columns
df_teams.head()

20 names for 20 columns


,team,games,min,pts,fg2,fg3,fg,ft,oreb,dreb,reb,ast,stl,tov,blk,blk_against,mt,pf,fd,val
0,MEINS AVENIDA,7,1400 200,"500 71,4",127/276 46%,"57/156 36,5%","184/432 42,6%","75/106 70,8%",56 8,"141 20,1","197 28,1","106 15,1",70 10,"99 14,1","10 1,4","16 2,3",0 0,"132 18,9","125 17,9","498 71,1"
1,CADI LA SEU,7,1400 200,"449 64,1","125/275 45,5%","40/137 29,2%",165/412 40%,"79/109 72,5%","58 8,3","185 26,4","243 34,7","86 12,3","67 9,6","102 14,6","29 4,1","8 1,1","2 0,3","115 16,4","116 16,6","496 70,9"
2,MIRAI EUSKOTREN,7,1425 203,"491 70,1","146/320 45,6%",36/109 33%,"182/429 42,4%","91/121 75,2%","71 10,1","146 20,9",217 31,"90 12,9","67 9,6","103 14,7","17 2,4","12 1,7",0 0,126 18,"135 19,3",511 73
3,INVERSUS GERNIKA BIZKAIA,7,1450 207,"541 77,3","155/296 52,4%",42/150 28%,"197/446 44,2%","105/153 68,6%","65 9,3",168 24,"233 33,3","103 14,7","62 8,9",105 15,"11 1,6","10 1,4",0 0,"124 17,7","136 19,4",560 80
4,CASADEMONT ZARAGOZA,7,1400 200,581 83,"139/263 52,9%","72/196 36,7%",211/459 46%,"87/120 72,5%","88 12,6","208 29,7","296 42,3","138 19,7","60 8,6","106 15,1","9 1,3","9 1,3",0 0,"127 18,1","138 19,7","708 101,1"


In [13]:
def get_total(cell):
    """'500 71,4' -> 500. Keeps the season total (the first number)."""
    first_part = str(cell).split()[0]   # split on spaces, keep the first piece
    return int(first_part)

def get_made_att(cell):
    """'127/276 46%' -> (127, 276). Returns made and attempted shots."""
    first_part = str(cell).split()[0]   # '127/276'
    made, att = first_part.split("/")   # '127' and '276'
    return int(made), int(att)

# Test both on one example before using them on the whole table
print(get_total("500 71,4"))
print(get_made_att("127/276 46%"))

500
(127, 276)


In [14]:
# 1. Count columns: keep only the season total
count_cols = ["min", "pts", "oreb", "dreb", "reb", "ast", "stl", "tov",
              "blk", "blk_against", "mt", "pf", "fd", "val"]

for col in count_cols:
    df_teams[col] = df_teams[col].apply(get_total)

# 2. Shooting columns: split into made and attempted
shot_cols = ["fg2", "fg3", "fg", "ft"]

for col in shot_cols:
    df_teams[col + "_made"] = df_teams[col].apply(lambda c: get_made_att(c)[0])
    df_teams[col + "_att"]  = df_teams[col].apply(lambda c: get_made_att(c)[1])

df_teams = df_teams.drop(columns=shot_cols)   # the original text columns are no longer needed

df_teams.head()

,team,games,min,pts,oreb,dreb,reb,ast,stl,tov,...,fd,val,fg2_made,fg2_att,fg3_made,fg3_att,fg_made,fg_att,ft_made,ft_att
0,MEINS AVENIDA,7,1400,500,56,141,197,106,70,99,...,125,498,127,276,57,156,184,432,75,106
1,CADI LA SEU,7,1400,449,58,185,243,86,67,102,...,116,496,125,275,40,137,165,412,79,109
2,MIRAI EUSKOTREN,7,1425,491,71,146,217,90,67,103,...,135,511,146,320,36,109,182,429,91,121
3,INVERSUS GERNIKA BIZKAIA,7,1450,541,65,168,233,103,62,105,...,136,560,155,296,42,150,197,446,105,153
4,CASADEMONT ZARAGOZA,7,1400,581,88,208,296,138,60,106,...,138,708,139,263,72,196,211,459,87,120


In [15]:
df_teams.dtypes

team             str
games          int64
min            int64
pts            int64
oreb           int64
dreb           int64
reb            int64
ast            int64
stl            int64
tov            int64
blk            int64
blk_against    int64
mt             int64
pf             int64
fd             int64
val            int64
fg2_made       int64
fg2_att        int64
fg3_made       int64
fg3_att        int64
fg_made        int64
fg_att         int64
ft_made        int64
ft_att         int64
dtype: object

### Data check
If the parsing worked, the numbers must agree with each other:
field goals = 2s + 3s, and points = 2·(2s made) + 3·(3s made) + free throws made.

In [16]:
check_fg  = df_teams["fg_made"] == df_teams["fg2_made"] + df_teams["fg3_made"]
check_pts = df_teams["pts"] == 2 * df_teams["fg2_made"] + 3 * df_teams["fg3_made"] + df_teams["ft_made"]

print("Field goals OK for every team:", check_fg.all())
print("Points OK for every team:", check_pts.all())

Field goals OK for every team: True
Points OK for every team: True


### Possessions
We estimate each team's possessions with the standard formula:
possessions = FGA − OREB + TOV + 0.44 × FTA.
We use them later to put every player on a per-100-possessions scale, as in the paper.

In [17]:
df_teams["poss"] = (df_teams["fg_att"] - df_teams["oreb"]
                    + df_teams["tov"] + 0.44 * df_teams["ft_att"])

df_teams["poss_per_game"] = df_teams["poss"] / df_teams["games"]   # pace

df_teams[["team", "games", "poss", "poss_per_game"]].sort_values("poss_per_game", ascending=False)

,team,games,poss,poss_per_game
3,INVERSUS GERNIKA BIZKAIA,7,553.32,79.045714
5,VALENCIA B.C.,7,545.08,77.868571
9,CELTA FEMXA ZORKA,7,537.40,76.771429
12,DURÁN MAQUINARIA ENSINO,7,530.68,75.811429
8,BAXI FERROL,7,529.96,75.708571
4,CASADEMONT ZARAGOZA,7,529.80,75.685714
0,MEINS AVENIDA,7,521.64,74.520000
15,MOVISTAR ESTUDIANTES,7,518.60,74.085714
11,KUTXABANK ARASKI,7,517.72,73.960000
2,MIRAI EUSKOTREN,7,514.24,73.462857


## 1.2 Team links
Each team name in the league table links to the team page, which has the roster
(position and height) and a link to every player's page.

In [18]:
tables_links = pd.read_html(StringIO(response.text), extract_links="body")
team_links = tables_links[0]

team_links.iloc[:3, 0]   # first 3 rows, first column

0    (MEINS AVENIDA, https://baloncestoenvivo.feb.e...
1    (CADI LA SEU, https://baloncestoenvivo.feb.es/...
2    (MIRAI EUSKOTREN, https://baloncestoenvivo.feb...
Name: (Unnamed: 0_level_0, Equipo), dtype: object

In [19]:
team_cells = team_links.iloc[:, 0]   # all rows, first column: (name, url) tuples

df_team_urls = pd.DataFrame({
    "team":     [cell[0] for cell in team_cells],   # position 0 of each tuple = name
    "team_url": [cell[1] for cell in team_cells],   # position 1 of each tuple = link
})

df_team_urls

,team,team_url
0,MEINS AVENIDA,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
1,CADI LA SEU,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
2,MIRAI EUSKOTREN,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
3,INVERSUS GERNIKA BIZKAIA,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
4,CASADEMONT ZARAGOZA,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
5,VALENCIA B.C.,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
6,AZULMARINO MALLORCA PALMA,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
7,HOZONO GLOBAL JAIRIS,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
8,BAXI FERROL,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...
9,CELTA FEMXA ZORKA,https://baloncestoenvivo.feb.es/Equipo.aspx?i=...


## 1.3 Roster of one team (test)
Before looping over the league, we test on one team: Meins Avenida.

In [20]:
test_url = df_team_urls.loc[0, "team_url"]   # first team: Meins Avenida
print(test_url)

response_team = requests.get(test_url, headers=headers)
print(response_team.status_code)

team_tables = pd.read_html(StringIO(response_team.text))
for i, t in enumerate(team_tables):
    print(i, t.shape)

https://baloncestoenvivo.feb.es/Equipo.aspx?i=1008381
200
0 (10, 9)


In [21]:
df_roster_test = team_tables[0]
df_roster_test.head()

,Unnamed: 0,Nombre,Puesto,Dorsal,Nacimiento,Nacionalidad,Formación,Altura,Peso
0,NaN,DAVINIA ANGEL VILLEN,A-PÍVOT,58,14/05/2004 Terrassa (Barcelona),ESPAÑA,SI,188,-
1,NaN,KENNEDY LAUREN BROWN,NaN,-,30/10/2000,ESTADOS UNIDOS,NO,-,-
2,NaN,MARTA CANELLA RODRIGUEZ,ESCOLTA,8,27/02/1995 Vigo (Pontevedra),ESPAÑA,SI,183,-
3,NaN,MARTINA FASSINA,ALERO,19,07/04/1999,ITALIA,NO,183,-
4,NaN,EMMA MELINA JOHANSSON,PÍVOT,22,28/08/2003,SUECIA,NO,192,-


In [22]:
roster_links = pd.read_html(StringIO(response_team.text), extract_links="body")[0]

name_cells = roster_links["Nombre"]   # (name, url) tuples

df_roster_test = pd.DataFrame({
    "player":      [cell[0] for cell in name_cells],
    "player_url":  [cell[1] for cell in name_cells],
    "position":    df_roster_test["Puesto"],
    "height":      df_roster_test["Altura"],
    "nationality": df_roster_test["Nacionalidad"],
})

df_roster_test

,player,player_url,position,height,nationality
0,DAVINIA ANGEL VILLEN,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,A-PÍVOT,188,ESPAÑA
1,KENNEDY LAUREN BROWN,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,NaN,-,ESTADOS UNIDOS
2,MARTA CANELLA RODRIGUEZ,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,ESCOLTA,183,ESPAÑA
3,MARTINA FASSINA,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,ALERO,183,ITALIA
4,EMMA MELINA JOHANSSON,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,PÍVOT,192,SUECIA
5,STEPHANIE MAWULI,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,ALERO,182,JAPON
6,REBEKA MIKULASIKOVA,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,PÍVOT,192,ESLOVAQUIA
7,SARA ROUMY,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,ALERO,181,FRANCIA
8,INES SANTIBAÑEZ HERRERO,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,BASE,173,ESPAÑA
9,CLAUDIA SORIANO PEREZ,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,BASE,170,ESPAÑA


## 1.4 Stats of one player (test)
Each player page has her season stats in two tables: averages ("Medias") and totals ("Totales").
We keep the totals and compute every rate ourselves.

In [23]:
player_url = df_roster_test.loc[0, "player_url"]
print(player_url)

response_player = requests.get(player_url, headers=headers)
print(response_player.status_code)

player_tables = pd.read_html(StringIO(response_player.text))
for i, t in enumerate(player_tables):
    print(i, t.shape)

https://baloncestoenvivo.feb.es/Jugador.aspx?i=1008381&c=2239381
200
0 (5, 6)
1 (3, 20)
2 (3, 20)


In [24]:
player_tables[2]

Unnamed: 0_level_0                  Unnamed: 1_level_0  \
                                 FASE                                Part   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                  LR                                   7   
2                                 NaN                                   7   

                   Unnamed: 2_level_0                  Unnamed: 3_level_0  \
                                  MIN                                  PT   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                               61:15                                  13   
2                               61:15                                  13   

                   Unnamed: 4_level_0                  Unnamed: 5_level_0  \
                                   T2                                  T3   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                5/14                                 1/3   
2                                5/14                                 1/3   

                   Unnamed: 6_level_0                  Unnamed: 7_level_0  \
                                   TC                                  TL   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                6/17                                 0/0   
2                                6/17                                 0/0   

                              Rebotes                                      \
                                   RO                                  RD   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                   4                                   5   
2                                   4                                   5   

                                                      Unnamed: 11_level_0  \
                                   RT                                  AS   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                   9                                   2   
2                                   9                                   2   

                  Unnamed: 12_level_0                 Unnamed: 13_level_0  \
                                   BR                                  BP   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                   2                                   5   
2                                   2                                   5   

                              Tapones                                      \
                                   TF                                  TC   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                   0                                   0   
2                                   0                                   0   

                  Unnamed: 16_level_0                              Faltas  \
                                   MT                                  FC   
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA   
1                                   0                                   8   
2                                   0                                   8   

                                                      Unnamed: 19_level_0  
                                   FR                                  VA  
0  Temp: 26/27. Equipo: MEINS AVENIDA  Temp: 26/27. Equipo: MEINS AVENIDA  
1                                   1                                   3  
2                                   1                                   3

In [25]:
def get_minutes(cell):
    """'61:15' -> 61.25. Converts 'minutes:seconds' into decimal minutes."""
    minutes, seconds = str(cell).split(":")
    return int(minutes) + int(seconds) / 60

print(get_minutes("61:15"))   # test: 61.25

61.25


In [26]:
df_p = player_tables[2].copy()   # copy: we change it without touching the original

# Same 20 columns as the team table; the first one is the phase instead of the team
df_p.columns = ["phase"] + clean_columns[1:]

# Keep only the regular season row (same competition as the team possessions)
df_p = df_p[df_p["phase"] == "LR"]

# Minutes: 'mm:ss' -> decimal minutes
df_p["min"] = df_p["min"].apply(get_minutes)

# Count columns: plain numbers (get_total also works on '13')
player_count_cols = [c for c in count_cols if c != "min"]
for col in player_count_cols:
    df_p[col] = df_p[col].apply(get_total)

# Shooting columns: '5/14' -> made and attempted
for col in shot_cols:
    df_p[col + "_made"] = df_p[col].apply(lambda c: get_made_att(c)[0])
    df_p[col + "_att"]  = df_p[col].apply(lambda c: get_made_att(c)[1])

df_p = df_p.drop(columns=shot_cols)

df_p

,phase,games,min,pts,oreb,dreb,reb,ast,stl,tov,...,fd,val,fg2_made,fg2_att,fg3_made,fg3_att,fg_made,fg_att,ft_made,ft_att
1,LR,7,61.25,13,4,5,9,2,2,5,...,1,3,5,14,1,3,6,17,0,0


## 1.5 One function per player
Everything that worked for one player goes into a function that receives the player URL
and returns one clean row. It returns None when there is nothing to keep
(page not available, or the player has no regular-season games yet).

In [29]:
def get_player_stats(player_url):
    """Download one player page and return her regular-season totals as one clean row.
    Returns None if the page fails or the player has no regular-season stats."""

    response = requests.get(player_url, headers=headers)
    if response.status_code != 200:              # page not available
        return None

    tables = pd.read_html(StringIO(response.text))
    if len(tables) < 3:                          # no stats tables: she has not played
        return None

    df = tables[2].copy()                        # 'Totales' table
    df.columns = ["phase"] + clean_columns[1:]
    df = df[df["phase"] == "LR"]                 # regular season only
    if df.empty:                                 # played, but not in the regular season
        return None

    df["min"] = df["min"].apply(get_minutes)
    for col in player_count_cols:
        df[col] = df[col].apply(get_total)
    for col in shot_cols:
        df[col + "_made"] = df[col].apply(lambda c: get_made_att(c)[0])
        df[col + "_att"]  = df[col].apply(lambda c: get_made_att(c)[1])

    df = df.drop(columns=shot_cols + ["phase"])   # 'phase' is always LR now
    df["player_url"] = player_url                 # key to join with the roster later
    return df.reset_index(drop=True)

In [28]:
print(get_player_stats(df_roster_test.loc[0, "player_url"]))   # Davinia: should return one row
print(get_player_stats(df_roster_test.loc[1, "player_url"]))   # Kennedy Brown: no position/height, let's see

  games    min  pts  oreb  dreb  reb  ast  stl  tov  blk  ...  val  fg2_made  \
0     7  61.25   13     4     5    9    2    2    5    0  ...    3         5   

   fg2_att  fg3_made  fg3_att  fg_made  fg_att  ft_made  ft_att  \
0       14         1        3        6      17        0       0   

                                          player_url  
0  https://baloncestoenvivo.feb.es/Jugador.aspx?i...  

[1 rows x 24 columns]
  games   min  pts  oreb  dreb  reb  ast  stl  tov  blk  ...  val  fg2_made  \
0     4  57.2   17     5     4    9    2    1    3    3  ...   15         7   

   fg2_att  fg3_made  fg3_att  fg_made  fg_att  ft_made  ft_att  \
0       12         0        0        7      12        3       4   

                                          player_url  
0  https://baloncestoenvivo.feb.es/Jugador.aspx?i...  

[1 rows x 24 columns]


## 1.6 One function per team
The roster code from 1.3 goes into a function too: it receives a team page and
returns one row per player with her link, position, height and nationality.

In [32]:
def get_roster(team_url, team_name):
    """Download one team page and return its roster: one row per player."""

    response = requests.get(team_url, headers=headers)
    roster = pd.read_html(StringIO(response.text))[0]                         # text
    links  = pd.read_html(StringIO(response.text), extract_links="body")[0]   # (text, url)

    return pd.DataFrame({
        "team":        team_name,
        "player":      [cell[0] for cell in links["Nombre"]],
        "player_url":  [cell[1] for cell in links["Nombre"]],
        "position":    roster["Puesto"],
        "height":      roster["Altura"],
        "nationality": roster["Nacionalidad"],
    })

In [33]:
import time   # to wait between requests

roster = get_roster(df_team_urls.loc[0, "team_url"], df_team_urls.loc[0, "team"])

player_rows = []                                  # empty list to collect one row per player

for url in roster["player_url"]:
    stats = get_player_stats(url)
    if stats is not None:                         # skip players with no regular-season stats
        player_rows.append(stats)
    time.sleep(1)                                 # 1 second between requests: be polite to the server

df_team_test = pd.concat(player_rows, ignore_index=True)        # stack the rows into one table
df_team_test = pd.merge(roster, df_team_test, on="player_url")  # add team, name, position, height

print(len(roster), "players in the roster,", len(df_team_test), "with stats")
df_team_test[["player", "position", "height", "games", "min", "pts"]]

10 players in the roster, 10 with stats


,player,position,height,games,min,pts
0,DAVINIA ANGEL VILLEN,A-PÍVOT,188,7,61.250000,13
1,KENNEDY LAUREN BROWN,NaN,-,4,57.200000,17
2,MARTA CANELLA RODRIGUEZ,ESCOLTA,183,6,123.266667,46
3,MARTINA FASSINA,ALERO,183,5,102.483333,41
4,EMMA MELINA JOHANSSON,PÍVOT,192,7,158.650000,53
5,STEPHANIE MAWULI,ALERO,182,7,214.000000,91
6,REBEKA MIKULASIKOVA,PÍVOT,192,7,181.050000,77
7,SARA ROUMY,ALERO,181,7,186.383333,54
8,INES SANTIBAÑEZ HERRERO,BASE,173,7,124.083333,28
9,CLAUDIA SORIANO PEREZ,BASE,170,7,185.916667,80


In [34]:
print("Sum of player points:", df_team_test["pts"].sum())
print("Team points (league table):", df_teams.loc[0, "pts"])

Sum of player points: 500
Team points (league table): 500


## 1.7 The whole league
One function per season: league page → team links → roster → player stats.
The season is the only thing that changes in the URL (t=2026 is 2026/27, t=2025 is 2025/26),
so the same code collects both seasons.

In [39]:
def get_league_players(season):
    """Collect the regular-season totals of every player in the Liga Femenina for one season.
    season: first year of the season, e.g. 2026 for 2026/27."""

    league_url = f"https://baloncestoenvivo.feb.es/estadisticas.aspx?g=4&t={season}&nm=lfendesa"
    response = requests.get(league_url, headers=headers)
    team_cells = pd.read_html(StringIO(response.text), extract_links="body")[0].iloc[:, 0]

    all_players = []                                       # one table per team

    for team_name, team_url in team_cells:                 # each cell is a (name, url) tuple
        try:
            roster = get_roster(team_url, team_name)

            player_rows = []
            for url in roster["player_url"]:
                stats = get_player_stats(url)
                if stats is not None:
                    player_rows.append(stats)
                time.sleep(1)                              # be polite to the server

            df_team = pd.merge(roster, pd.concat(player_rows, ignore_index=True), on="player_url")
            all_players.append(df_team)
            print(f"{team_name}: {len(df_team)} players")  # progress

        except Exception as e:
            print(f"ERROR in {team_name}: {e}")            # say which team failed and why, keep going

    df = pd.concat(all_players, ignore_index=True)
    df["season"] = season
    return df
    
# First attempt, kept to show the error (output below). Fixed in section 1.8.
# df_2627 = get_league_players(2026)
# print(df_2627.shape)
# df_2627.to_csv("data/players_2026_27.csv", index=False)

In [36]:
df_2627 = get_league_players(2026)

print(df_2627.shape)
df_2627.to_csv("data/players_2026_27.csv", index=False)

MEINS AVENIDA: 10 players
CADI LA SEU: 11 players
MIRAI EUSKOTREN: 10 players
INVERSUS GERNIKA BIZKAIA: 10 players
ERROR in CASADEMONT ZARAGOZA: not enough values to unpack (expected 2, got 1)
ERROR in VALENCIA B.C.: not enough values to unpack (expected 2, got 1)
ERROR in AZULMARINO MALLORCA PALMA: not enough values to unpack (expected 2, got 1)
ERROR in HOZONO GLOBAL JAIRIS: not enough values to unpack (expected 2, got 1)
BAXI FERROL: 10 players
CELTA FEMXA ZORKA: 10 players
ERROR in SPAR GIRONA: not enough values to unpack (expected 2, got 1)
KUTXABANK ARASKI: 10 players
ERROR in DURÁN MAQUINARIA ENSINO: not enough values to unpack (expected 2, got 1)
ERROR in IMPULSO 360 ESTEPONA: not enough values to unpack (expected 2, got 1)
ERROR in INNOVA-TSN LEGANÉS: not enough values to unpack (expected 2, got 1)
MOVISTAR ESTUDIANTES: 11 players
(82, 30)


### Debugging
Eight teams failed with the same error. We look for the exact player and cell that breaks the parser.

In [37]:
roster_zgz = get_roster(df_team_urls.loc[4, "team_url"], df_team_urls.loc[4, "team"])   # Casademont Zaragoza

for i, url in enumerate(roster_zgz["player_url"]):
    try:
        get_player_stats(url)
    except Exception as e:
        print("Player with the problem:", roster_zgz.loc[i, "player"])
        print("URL:", url)
        print("Error:", e)
        problem_url = url
        break                       # stop at the first one, that is enough to investigate
    time.sleep(1)

Player with the problem: LAIA FLORES COSTA
URL: https://baloncestoenvivo.feb.es/Jugador.aspx?i=1008630&c=1572736
Error: not enough values to unpack (expected 2, got 1)


In [38]:
r = requests.get(problem_url, headers=headers)
pd.read_html(StringIO(r.text))[2]

Unnamed: 0_level_0  \
                                       FASE   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                        LR   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                        SP   
4                                       NaN   

                         Unnamed: 1_level_0  \
                                       Part   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                         7   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                         1   
4                                         8   

                         Unnamed: 2_level_0  \
                                        MIN   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                     17:50   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                     14:24   
4                                     17:24   

                         Unnamed: 3_level_0  \
                                         PT   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                         5   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                         9   
4                                        55   

                         Unnamed: 4_level_0  \
                                         T2   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                     31,3%   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                       50%   
4                                     36,4%   

                         Unnamed: 5_level_0  \
                                         T3   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                     36,8%   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                     33,3%   
4                                     36,4%   

                         Unnamed: 6_level_0  \
                                         TC   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                     34,3%   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                     44,4%   
4                                     36,4%   

                         Unnamed: 7_level_0  \
                                         TL   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                     66,7%   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                        0%   
4                                     66,7%   

                                    Rebotes  \
                                         RO   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                        03   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                         2   
4                                        05   

                                             \
                                         RD   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                        21   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                         2   
4                                        21   

                                             \
                                         RT   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                        24   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                         4   
4                                        26   

                        Unnamed: 11_level_0  \
                                         AS   
0  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
1                                        26   
2  Temp: 26/27. Equipo: CASADEMONT ZARAGOZA   
3                                         1   
4                                        24   

                        Unnamed: 12_level_0  \
                  

## 1.8 Fixing the parser
**What went wrong:** 8 of 16 teams failed. Players who also played the Supercopa have extra rows,
so the tables move on the page and `tables[2]` was the averages table (e.g. `31,3%`), not the totals.

**Fix 1:** find the totals table by its content (20 columns and `made/attempted` values), not by its position.

**Fix 2:** protect each player with its own `try/except`, so one bad page loses one player, not the whole team.

In [48]:
def get_player_stats(player_url):
    """Download one player page and return her regular-season totals as one clean row.
    Returns None if the page fails or the player has no regular-season stats."""

    response = requests.get(player_url, headers=headers)
    if response.status_code != 200:              # page not available
        return None

    tables = pd.read_html(StringIO(response.text))

    # NEW: find the totals table by its content, not by its position on the page:
    # 20 columns, and the 2-point column (5th) has 'made/attempted' values like '5/14'
    df = None
    for t in tables:
        if t.shape[1] == 20 and t.iloc[:, 4].astype(str).str.contains("/").any():
            df = t.copy()
    if df is None:                               # no totals table: she has not played
        return None

    df.columns = ["phase"] + clean_columns[1:]
    df = df[df["phase"] == "LR"]                 # regular season only
    if df.empty:                                 # played, but not in the regular season
        return None

    df["min"] = df["min"].apply(get_minutes)
    for col in player_count_cols:
        df[col] = df[col].apply(get_total)
    for col in shot_cols:
        df[col + "_made"] = df[col].apply(lambda c: get_made_att(c)[0])
        df[col + "_att"]  = df[col].apply(lambda c: get_made_att(c)[1])

    df = df.drop(columns=shot_cols + ["phase"])   # 'phase' is always LR now
    df["player_url"] = player_url                 # key to join with the roster later
    return df.reset_index(drop=True)

In [49]:
get_player_stats(problem_url)   # Laia Flores: should now return one row

,games,min,pts,oreb,dreb,reb,ast,stl,tov,blk,...,val,fg2_made,fg2_att,fg3_made,fg3_att,fg_made,fg_att,ft_made,ft_att,player_url
0,7,124.85,35,2,15,17,18,7,8,0,...,45,5,16,7,19,12,35,4,6,https://baloncestoenvivo.feb.es/Jugador.aspx?i...


In [50]:
def get_league_players(season):
    """Collect the regular-season totals of every player in the Liga Femenina for one season.
    season: first year of the season, e.g. 2026 for 2026/27."""

    league_url = f"https://baloncestoenvivo.feb.es/estadisticas.aspx?g=4&t={season}&nm=lfendesa"
    response = requests.get(league_url, headers=headers)
    team_cells = pd.read_html(StringIO(response.text), extract_links="body")[0].iloc[:, 0]

    all_players = []                                       # one table per team

    for team_name, team_url in team_cells:                 # each cell is a (name, url) tuple
        try:
            roster = get_roster(team_url, team_name)

            player_rows = []
            for url in roster["player_url"]:
                try:                                                  # NEW: protect each player
                    stats = get_player_stats(url)
                    if stats is not None:
                        player_rows.append(stats)
                except Exception as e:
                    print(f"   ERROR in player {url}: {e}")          # NEW: lose one player, not the team
                time.sleep(1)                              # be polite to the server

            df_team = pd.merge(roster, pd.concat(player_rows, ignore_index=True), on="player_url")
            all_players.append(df_team)
            print(f"{team_name}: {len(df_team)} players")  # progress

        except Exception as e:
            print(f"ERROR in {team_name}: {e}")            # say which team failed and why, keep going

    df = pd.concat(all_players, ignore_index=True)
    df["season"] = season
    return df

In [51]:
df_2627 = get_league_players(2026)

print(df_2627.shape)
df_2627.to_csv("data/players_2026_27.csv", index=False)

MEINS AVENIDA: 10 players
CADI LA SEU: 11 players
MIRAI EUSKOTREN: 10 players
INVERSUS GERNIKA BIZKAIA: 10 players
CASADEMONT ZARAGOZA: 12 players
VALENCIA B.C.: 12 players
AZULMARINO MALLORCA PALMA: 11 players
HOZONO GLOBAL JAIRIS: 10 players
BAXI FERROL: 10 players
CELTA FEMXA ZORKA: 10 players
SPAR GIRONA: 12 players
KUTXABANK ARASKI: 10 players
DURÁN MAQUINARIA ENSINO: 11 players
IMPULSO 360 ESTEPONA: 12 players
INNOVA-TSN LEGANÉS: 11 players
MOVISTAR ESTUDIANTES: 11 players
(173, 30)


### Data check: players vs teams
If no player is missing, the points of each team's players must add up to the team's points.

In [52]:
pts_players = df_2627.groupby("team")["pts"].sum()   # points per team, adding up its players
pts_teams   = df_teams.set_index("team")["pts"]       # points per team, from the league table

check = pd.DataFrame({"players": pts_players, "team": pts_teams})
check["difference"] = check["team"] - check["players"]

check.sort_values("difference", ascending=False)

,players,team,difference
team,,,
VALENCIA B.C.,472,575,103
DURÁN MAQUINARIA ENSINO,407,426,19
BAXI FERROL,508,510,2
AZULMARINO MALLORCA PALMA,547,547,0
CADI LA SEU,449,449,0
CASADEMONT ZARAGOZA,581,581,0
CELTA FEMXA ZORKA,461,461,0
HOZONO GLOBAL JAIRIS,508,508,0
IMPULSO 360 ESTEPONA,469,469,0


### Who is missing?
Valencia's players add up to 103 points less than the team. We check whether the missing player
is in the roster (and our function skipped her) or no longer in the roster.

In [53]:
team_row = df_team_urls[df_team_urls["team"] == "VALENCIA B.C."].iloc[0]
roster_vbc = get_roster(team_row["team_url"], team_row["team"])

collected = df_2627["player_url"]                                  # players we have stats for
not_collected = roster_vbc[~roster_vbc["player_url"].isin(collected)]

print(len(roster_vbc), "players in the roster,", len(not_collected), "without stats")
not_collected[["player", "position", "player_url"]]

12 players in the roster, 0 without stats


,player,position,player_url


## 1.9 From rosters to box scores
The roster check showed 103 missing points at Valencia: the roster page only lists current players,
so players who left the club are lost. Game box scores include everyone who played.
We follow the method from Class 9: one function per game, test it, then loop over the season.

In [54]:
calendar_url = "https://baloncestoenvivo.feb.es/calendario.aspx?g=4&t=2026&nm=lfendesa"
response_cal = requests.get(calendar_url, headers=headers)
print(response_cal.status_code)

cal_tables = pd.read_html(StringIO(response_cal.text))
print("Tables:", len(cal_tables))
for i, t in enumerate(cal_tables[:3]):          # look at the first 3 only
    print(i, t.shape)

cal_tables[0]

200
Tables: 30
0 (8, 3)
1 (8, 3)
2 (8, 3)


,Local,Resultado,Visitante
0,AZULMARINO MALLORCA PALMA,88-71,VALENCIA B.C.
1,BAXI FERROL,82-77,KUTXABANK ARASKI
2,DURÁN MAQUINARIA ENSINO,53-72,MIRAI EUSKOTREN
3,MOVISTAR ESTUDIANTES,82-79,SPAR GIRONA
4,INNOVA-TSN LEGANÉS,66-63,HOZONO GLOBAL JAIRIS
5,CELTA FEMXA ZORKA,80-84,INVERSUS GERNIKA BIZKAIA
6,MEINS AVENIDA,73-57,CADI LA SEU
7,CASADEMONT ZARAGOZA,68-41,IMPULSO 360 ESTEPONA


In [55]:
cal_links = pd.read_html(StringIO(response_cal.text), extract_links="body")   # (text, url) in every cell

game_rows = []

for round_number, t in enumerate(cal_links, start=1):    # one table per round: round 1, 2, 3...
    for _, row in t.iterrows():                           # one row per game
        result_text, game_url = row.iloc[1]               # 'Resultado' cell: (text, url)
        game_rows.append({
            "round":    round_number,
            "home":     row.iloc[0][0],                   # text only
            "away":     row.iloc[2][0],
            "result":   result_text,
            "game_url": game_url,
        })

df_games = pd.DataFrame(game_rows)

# A game was played if its result looks like '88-71' (numbers, dash, numbers)
df_games["played"] = df_games["result"].str.fullmatch(r"\d+-\d+")

print(len(df_games), "games in the calendar,", df_games["played"].sum(), "already played")
df_games[df_games["played"]].tail()

240 games in the calendar, 56 already played


,round,home,away,result,game_url,played
51,7,DURÁN MAQUINARIA ENSINO,INVERSUS GERNIKA BIZKAIA,66-72,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True
52,7,MOVISTAR ESTUDIANTES,CASADEMONT ZARAGOZA,67-68,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True
53,7,INNOVA-TSN LEGANÉS,BAXI FERROL,64-66,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True
54,7,CELTA FEMXA ZORKA,KUTXABANK ARASKI,75-56,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True
55,7,MEINS AVENIDA,MIRAI EUSKOTREN,70-74,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True


## 1.10 Box score of one game (test)
We test on one game we can check by hand: Spar Girona 78-80 Valencia B.C. (round 7).

In [56]:
test_game = df_games[(df_games["home"] == "SPAR GIRONA") & (df_games["round"] == 7)].iloc[0]
print(test_game["home"], test_game["result"], test_game["away"])
print(test_game["game_url"])

response_game = requests.get(test_game["game_url"], headers=headers)
print(response_game.status_code)

game_tables = pd.read_html(StringIO(response_game.text))
for i, t in enumerate(game_tables):
    print(i, t.shape)

SPAR GIRONA 78-80 VALENCIA B.C.
https://baloncestoenvivo.feb.es/Partido.aspx?p=2522824
200
0 (11, 22)
1 (12, 22)


In [57]:
game_tables[1]

Unnamed: 0_level_0 Unnamed: 1_level_0          Unnamed: 2_level_0  \
                    I                  D                     Jugador   
0                   *                1.0      PETERSON, ALEXIS GANAY   
1                 NaN                2.0   TAKACS-KISS, VIRAG CSILLA   
2                 NaN                4.0    BALOGUN, ELIZABETH BUMNI   
3                   *                6.0    BUENAVIDA ESTEVEZ, ELENA   
4                   *                8.0         PEREZ ARAUJO, MARIA   
5                 NaN               11.0  BORKOWSKA, KAMILA WERONIKA   
6                 NaN               13.0       PUEYO MELCHOR, HELENA   
7                   *               14.0    CARRERA QUINTANA, RAQUEL   
8                   *               15.0         BIBBY, CHLOE LOUISE   
9                 NaN               17.0        BLEDA ORTEGA, MARINA   
10                NaN               70.0     HERNANDEZ LOPEZ, ISABEL   
11                NaN                NaN                         NaN   

   Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0  \
                  MIN                 PT                 T2   
0               25:57                  2             0/1 0%   
1               20:18                  6            3/5 60%   
2               22:00                  5            1/4 25%   
3               09:38                  3           1/1 100%   
4               33:22                  6            1/2 50%   
5               08:28                  0             0/1 0%   
6               32:47                 15          6/9 66,7%   
7               27:52                 16         7/12 58,3%   
8               31:50                 25            2/5 40%   
9               03:10                  2           1/1 100%   
10              09:38                  0             0/1 0%   
11             225:00                 80        22/42 52,4%   

   Unnamed: 6_level_0 Unnamed: 7_level_0 Unnamed: 8_level_0 Rebotes  ...  \
                   T3                 TC                 TL      RO  ...   
0              0/2 0%             0/3 0%           2/2 100%       0  ...   
1              0/0 0%            3/5 60%             0/0 0%       0  ...   
2             1/2 50%          2/6 33,3%             0/2 0%       1  ...   
3              0/0 0%           1/1 100%           1/1 100%       0  ...   
4             1/4 25%          2/6 33,3%            1/2 50%       1  ...   
5              0/1 0%             0/2 0%             0/0 0%       1  ...   
6              0/3 0%           6/12 50%           3/3 100%       0  ...   
7              0/2 0%           7/14 50%          2/3 66,7%       2  ...   
8           4/6 66,7%         6/11 54,5%           9/10 90%       1  ...   
9              0/0 0%           1/1 100%             0/0 0%       0  ...   
10             0/1 0%             0/2 0%             0/0 0%       1  ...   
11         6/21 28,6%        28/63 44,4%        18/23 78,3%       7  ...   

   Unnamed: 12_level_0 Unnamed: 13_level_0 Unnamed: 14_level_0 Tapones     \
                    AS                  BR                  BP      TF TC   
0                    0                   0                   5       0  0   
1                    0                   1                   2       0  0   
2                    1                   1                   1       0  0   
3                    0                   0                   0       0  0   
4                    1                   2                   0       0  1   
5                    0                   1                   1       1  0   
6                    3                   2                   4       2  0   
7                    1                   1                   2       0  1   
8                    0                   1                   2       0  0   
9                    0                   1                   0       0  0   
10                   3                   0                   1       0  0   
11                   9                  10   

In [58]:
# Box score columns: starter, number, name, the usual 18 stats, plus-minus
box_columns = ["starter", "number", "player"] + clean_columns[2:] + ["plus_minus"]

def get_game_boxscore(game):
    """Download one game page and return one row per player who played in it.
    game: one row of df_games (round, home, away, game_url)."""

    response = requests.get(game["game_url"], headers=headers)
    tables = pd.read_html(StringIO(response.text))                         # text
    links  = pd.read_html(StringIO(response.text), extract_links="body")   # (text, url)

    teams = [(game["home"], game["away"]), (game["away"], game["home"])]   # table 0 = home, table 1 = away
    game_rows = []

    for k in [0, 1]:
        df = tables[k].copy()
        df.columns = box_columns
        df["player_url"] = [cell[1] for cell in links[k].iloc[:, 2]]      # link of the 'Jugador' cell
        df = df[df["player_url"].notna()]                                  # drops the team totals row

        df["team"], df["opponent"] = teams[k]
        df["round"] = game["round"]
        df["game_url"] = game["game_url"]
        df["starter"] = df["starter"] == "*"                               # True / False

        df["min"] = df["min"].apply(get_minutes)
        for col in player_count_cols:
            df[col] = df[col].apply(get_total)
        for col in shot_cols:
            df[col + "_made"] = df[col].apply(lambda c: get_made_att(c)[0])
            df[col + "_att"]  = df[col].apply(lambda c: get_made_att(c)[1])
        df["plus_minus"] = pd.to_numeric(df["plus_minus"], errors="coerce")

        game_rows.append(df.drop(columns=shot_cols + ["number"]))

    return pd.concat(game_rows, ignore_index=True)

In [59]:
box_test = get_game_boxscore(test_game)

print(box_test.groupby("team")["pts"].sum())   # must be 78 Girona, 80 Valencia
box_test[["team", "player", "starter", "min", "pts", "fg3_made", "fg3_att"]].head(12)

team
SPAR GIRONA      78
VALENCIA B.C.    80
Name: pts, dtype: int64


,team,player,starter,min,pts,fg3_made,fg3_att
0,SPAR GIRONA,"ONYIAH, UGONNE MICHELLE",True,22.650000,11,0,0
1,SPAR GIRONA,"CONNER, MADISON ELIZABETH",False,32.666667,14,4,12
2,SPAR GIRONA,"MARTIN CARRION, IYANA",True,34.433333,17,3,10
3,SPAR GIRONA,"GUERRERO ALCANTARA, CAROLINA",True,39.283333,15,1,1
4,SPAR GIRONA,"SENYUREK ARSLAN, TILBE",False,28.066667,9,1,2
5,SPAR GIRONA,"CASAS CARRERAS, QUERALT",True,19.916667,2,0,1
6,SPAR GIRONA,"CAMILION CUCCARO, JUANA",True,37.416667,10,1,5
7,SPAR GIRONA,"LOPEZ RODRIGUEZ, AINHOA",False,2.866667,0,0,0
8,SPAR GIRONA,"LEKOVIC, MARIJA",False,4.500000,0,0,0
9,SPAR GIRONA,"BOTA GARCIA, MARIONA",False,3.200000,0,0,1


## 1.11 The whole season, game by game
We run the box score function for every played game and stack the results:
one row per player per game.

In [60]:
played_games = df_games[df_games["played"]]

box_list = []                                              # one table per game

for _, game in played_games.iterrows():
    try:
        box_list.append(get_game_boxscore(game))
    except Exception as e:
        print(f"ERROR in round {game['round']}, {game['home']} - {game['away']}: {e}")
    time.sleep(1)                                          # be polite to the server

    if game.name % 8 == 7:                                 # last game of each round
        print(f"Round {game['round']} done")

df_box_2627 = pd.concat(box_list, ignore_index=True)

print(df_box_2627.shape)
df_box_2627.to_csv("data/boxscores_2026_27.csv", index=False)

Round 1 done
Round 2 done
Round 3 done
Round 4 done
Round 5 done
Round 6 done
Round 7 done
(1124, 30)


In [61]:
pts_box   = df_box_2627.groupby("team")["pts"].sum()
check_box = pd.DataFrame({"box_scores": pts_box, "team": df_teams.set_index("team")["pts"]})
check_box["difference"] = check_box["team"] - check_box["box_scores"]

check_box.sort_values("difference", ascending=False).head()

,box_scores,team,difference
team,,,
AZULMARINO MALLORCA PALMA,547,547,0
BAXI FERROL,510,510,0
CADI LA SEU,449,449,0
CASADEMONT ZARAGOZA,581,581,0
CELTA FEMXA ZORKA,461,461,0


In [62]:
print("Every team matches:", (check_box["difference"] == 0).all())

Every team matches: True


**Result:** the box scores add up to the official team points for all 16 teams (difference = 0).
The roster-based method missed 103 points at Valencia; the game-by-game method misses nothing.

## 1.12 From games to season totals
One row per player: we add up her games. The player URL is the key (two players can share a name).

In [63]:
# Columns we add up over the season
sum_cols = ["min", "pts", "oreb", "dreb", "reb", "ast", "stl", "tov", "blk", "blk_against",
            "pf", "fd", "val", "plus_minus",
            "fg2_made", "fg2_att", "fg3_made", "fg3_att", "fg_made", "fg_att", "ft_made", "ft_att"]

# 1. Who she is and how much she played
info = df_box_2627.groupby("player_url").agg(
    player=("player", "first"),
    team=("team", "last"),            # last team, in case she changed clubs
    games=("game_url", "nunique"),    # number of different games
    starts=("starter", "sum"),        # True counts as 1
).reset_index()

# 2. Her season totals
totals = df_box_2627.groupby("player_url")[sum_cols].sum().reset_index()

# 3. Both side by side, joined by the player URL
df_players_2627 = pd.merge(info, totals, on="player_url")

print(df_players_2627.shape)
df_players_2627.sort_values("pts", ascending=False).head(10)

(192, 27)


,player_url,player,team,games,starts,min,pts,oreb,dreb,reb,...,val,plus_minus,fg2_made,fg2_att,fg3_made,fg3_att,fg_made,fg_att,ft_made,ft_att
107,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"DOOGAN, MARGARET CHRISTINE",BAXI FERROL,7,7,220.033333,127,6,35,41,...,127,-10.0,23,50,22,46,45,96,15,18
84,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"HOLM, KLARA MARIA SUSANNE",AZULMARINO MALLORCA PALMA,7,7,222.650000,124,11,45,56,...,199,79.0,28,56,13,24,41,80,29,34
81,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"ALEXANDER, KAYLA JANINE",AZULMARINO MALLORCA PALMA,7,7,208.050000,113,24,34,58,...,149,87.0,48,76,0,1,48,77,17,17
40,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"VONLEH, AARONETTE",INVERSUS GERNIKA BIZKAIA,7,6,172.483333,105,13,21,34,...,124,20.0,45,66,0,0,45,66,15,25
116,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"RONSIEK, EMMA GRACE",CELTA FEMXA ZORKA,7,7,230.416667,96,4,26,30,...,85,-65.0,22,50,12,37,34,87,16,22
47,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"SHEPPARD, AISHA MARIE",CASADEMONT ZARAGOZA,7,4,137.000000,95,3,11,14,...,84,87.0,13,25,18,33,31,58,15,16
33,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"SALVADORES ALVAREZ, ANGELA",INVERSUS GERNIKA BIZKAIA,7,7,231.866667,95,6,30,36,...,99,-25.0,19,51,10,38,29,89,27,35
85,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"TORRENS SALOM, ALBA",AZULMARINO MALLORCA PALMA,7,7,191.950000,94,2,21,23,...,85,78.0,26,51,7,31,33,82,21,23
5,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"MAWULI, STEPHANIE",MEINS AVENIDA,7,7,214.000000,91,12,24,36,...,116,-25.0,15,38,11,31,26,69,28,36
30,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"ROBINSON, PAIGE",MIRAI EUSKOTREN,7,6,203.300000,90,0,18,18,...,98,31.0,23,33,12,31,35,64,8,10


## 1.13 Height, position and nationality
These come from the team rosters collected in 1.7. We join them by player URL with a left join:
every player with box score stats is kept, even if she is no longer in a roster.

In [64]:
roster_info = df_2627[["player_url", "position", "height", "nationality"]]

df_players_2627 = pd.merge(df_players_2627, roster_info, on="player_url", how="left")

# Height comes as text ('188', or '-' when the FEB has no data) -> number, '-' becomes NaN
df_players_2627["height"] = pd.to_numeric(df_players_2627["height"], errors="coerce")

missing = df_players_2627[df_players_2627["height"].isna()]
print(len(missing), "players without height")
missing[["player", "team", "games", "min", "position"]].sort_values("min", ascending=False)

27 players without height


,player,team,games,min,position
22,"SONTAG, LINA MERLE",CADI LA SEU,5,123.516667,NaN
42,"MCSPADDEN, LARA GWENETH",INVERSUS GERNIKA BIZKAIA,7,113.400000,PÍVOT
58,"SUAREZ RODRIGUEZ, MARTA",VALENCIA B.C.,5,92.683333,NaN
153,"YURKEVICHUS, TETIANA",DURÁN MAQUINARIA ENSINO,5,79.366667,NaN
65,"HERNANDEZ LOPEZ, ISABEL",VALENCIA B.C.,7,76.316667,NaN
73,"CARRERA QUINTANA, RAQUEL",VALENCIA B.C.,3,69.866667,NaN
11,"BROWN, KENNEDY LAUREN",MEINS AVENIDA,4,57.200000,NaN
131,"ONYIAH, UGONNE MICHELLE",SPAR GIRONA,3,56.833333,NaN
155,"SAKEVICIUTE, LIVIJA",DURÁN MAQUINARIA ENSINO,3,50.566667,NaN
151,"WILLIAMS, PRISCILLA TRUDELL",DURÁN MAQUINARIA ENSINO,5,43.566667,NaN


In [65]:
# The same player in both tables: compare her two URLs
name = "CARRERA"
print("Box score:", df_players_2627[df_players_2627["player"].str.contains(name)]["player_url"].values)
print("Roster:   ", df_2627[df_2627["player"].str.contains(name)]["player_url"].values)

Box score: <StringArray>
['https://baloncestoenvivo.feb.es/Jugador.aspx?i=1008686&c=2912506',
 'https://baloncestoenvivo.feb.es/Jugador.aspx?i=1008963&c=1091338']
Length: 2, dtype: str
Roster:    <StringArray>
['https://baloncestoenvivo.feb.es/Jugador.aspx?i=1008686&c=2912506',
 'https://baloncestoenvivo.feb.es/Jugador.aspx?i=1008963&c=1091338']
Length: 2, dtype: str


In [66]:
in_roster = missing["player_url"].isin(df_2627["player_url"])

print("In a roster, but the FEB has no height:", in_roster.sum())
print("Not in any current roster (left the club):", (~in_roster).sum())

In a roster, but the FEB has no height: 8
Not in any current roster (left the club): 19


In [67]:
df_players_2627.to_csv("data/players_2026_27_from_boxscores.csv", index=False)
df_box_2627.to_csv("data/boxscores_2026_27.csv", index=False)
print("Saved")

Saved


**Missing heights:** 27 of 192 players have no height in the FEB data (missing in the roster, or the player left the club).
Only 5 of them pass the minutes filter; their heights will be filled by hand from official sources, with the source recorded.

In [68]:
pos = response_cal.text.find("2025/2026")
print(response_cal.text[pos - 1500 : pos + 300])

wrapper-seccion">
            
            
         
    

<div class="wrapper-tituloSeccion">
    <div class="tituloSeccion">
        <div id="_ctl0_MainContentPlaceHolderMaster_ControlTituloSeccion_logoCategoriaPanel" class="wrapper-logo">
	<img id="logoCategoriaImage" src="https://imagenes.feb.es/Imagen.aspx?g=4&amp;t=2026" border="0" />
</div>
        <div class="wrapper-text">
            <span class="titulo">Calendario</span>
            <span class="liga">LF ENDESA </span>
            <span class="temporada">2026/2027</span>
        </div>
    </div>
</div>


    <div class="wrapper-seccion-contents" id="page-calendario">
		<!-- PUBLICIDAD SETUPAD Body Tag: responsive top | without lazy-loading | single use per page -->
		<div class="publicidad">
			<div id="baloncestoenvivo_feb_es_responsive_top" class="baloncestoenvivo_feb_es_responsive_top display-flex">
				<script>
				  googletag.cmd.push(function() { googletag.display('baloncestoenvivo_feb_es_responsive_top'); });
				</

## 1.14 Previous season
The season is not chosen in the URL: the page is an ASP.NET form, and choosing a season in the
dropdown sends the form back to the server (a POST request). We do the same from Python:
read the form fields, change the season, and send the form.

In [69]:
import lxml.html

def get_calendar_page(season):
    """Return the HTML of the Liga Femenina calendar for one season (e.g. 2025 for 2025/26)."""
    session = requests.Session()                           # keeps the cookies between the two requests
    first = session.get(calendar_url, headers=headers)     # 1. the page as it comes (current season)

    form = lxml.html.fromstring(first.text).forms[0]       # 2. the form inside the page
    fields = dict(form.form_values())                      #    every field and its current value

    # 3. what the browser sends when you pick a season in the dropdown
    fields["_ctl0:MainContentPlaceHolderMaster:temporadasDropDownList"] = str(season)
    fields["__EVENTTARGET"] = "_ctl0$MainContentPlaceHolderMaster$temporadasDropDownList"
    fields["__EVENTARGUMENT"] = ""

    second = session.post(calendar_url, data=fields, headers=headers)   # 4. send the form back
    return second.text

html_2526 = get_calendar_page(2025)

pos = html_2526.find('class="temporada"')
print(html_2526[pos : pos + 50])                           # should say 2025/2026

class="temporada">2025/2026</span>
        </div>


In [70]:
def get_games(calendar_html):
    """From the calendar page, return one row per game: round, teams, result, link, played."""
    cal_links = pd.read_html(StringIO(calendar_html), extract_links="body")
    game_rows = []
    for round_number, t in enumerate(cal_links, start=1):
        for _, row in t.iterrows():
            result_text, game_url = row.iloc[1]
            game_rows.append({
                "round":    round_number,
                "home":     row.iloc[0][0],
                "away":     row.iloc[2][0],
                "result":   result_text,
                "game_url": game_url,
            })
    df = pd.DataFrame(game_rows)
    df["played"] = df["result"].str.fullmatch(r"\d+-\d+")
    return df


def get_season_boxscores(df_games):
    """Download the box score of every played game and stack them: one row per player per game."""
    box_list = []
    played = df_games[df_games["played"]]
    for _, game in played.iterrows():
        try:
            box_list.append(get_game_boxscore(game))
        except Exception as e:
            print(f"ERROR in round {game['round']}, {game['home']} - {game['away']}: {e}")
        time.sleep(1)
        if game.name % 8 == 7:
            print(f"Round {game['round']} done")
    return pd.concat(box_list, ignore_index=True)

In [71]:
df_games_2526 = get_games(html_2526)
print(len(df_games_2526), "games,", df_games_2526["played"].sum(), "played")   # should be 240 and 240

box_one = get_game_boxscore(df_games_2526.iloc[0])
print(df_games_2526.iloc[0]["home"], df_games_2526.iloc[0]["result"], df_games_2526.iloc[0]["away"])
print(box_one.groupby("team")["pts"].sum())

240 games, 240 played
KUTXABANK ARASKI 57-56 PERFUMERIAS AVENIDA
team
KUTXABANK ARASKI       57
PERFUMERIAS AVENIDA    56
Name: pts, dtype: int64


In [72]:
df_box_2526 = get_season_boxscores(df_games_2526)

print(df_box_2526.shape)
df_box_2526.to_csv("data/boxscores_2025_26.csv", index=False)

Round 1 done
Round 2 done
Round 3 done
Round 4 done
Round 5 done
Round 6 done
Round 7 done
Round 8 done
Round 9 done
Round 10 done
Round 11 done
Round 12 done
Round 13 done
Round 14 done
Round 15 done
Round 16 done
Round 17 done
Round 18 done
Round 19 done
Round 20 done
Round 21 done
Round 22 done
Round 23 done
Round 24 done
Round 25 done
Round 26 done
Round 27 done
Round 28 done
Round 29 done
Round 30 done
(4659, 30)


## 1.15 Data check, game by game
For every game, the points of each team's players must equal the official result in the calendar.

In [73]:
def check_games(df_games, df_box):
    """Compare, game by game, the box score points with the official result. Returns the games that do not match."""
    played = df_games[df_games["played"]].copy()

    scores = played["result"].str.split("-", expand=True).astype(int)   # '88-71' -> 88 and 71
    played["home_pts"] = scores[0]
    played["away_pts"] = scores[1]

    box_pts = df_box.groupby(["game_url", "team"])["pts"].sum()          # points per game and team

    played["home_box"] = [box_pts.get((url, team)) for url, team in zip(played["game_url"], played["home"])]
    played["away_box"] = [box_pts.get((url, team)) for url, team in zip(played["game_url"], played["away"])]

    wrong = (played["home_box"] != played["home_pts"]) | (played["away_box"] != played["away_pts"])
    print(f"{len(played)} games checked, {wrong.sum()} do not match")
    return played[wrong]

check_games(df_games, df_box_2627)        # 2026/27
check_games(df_games_2526, df_box_2526)   # 2025/26

56 games checked, 0 do not match
240 games checked, 30 do not match


,round,home,away,result,game_url,played,home_pts,away_pts,home_box,away_box
1,1,LOINTEK GERNIKA BIZKAIA,MOVISTAR ESTUDIANTES,89-88,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,89,88,89,68
12,2,MOVISTAR ESTUDIANTES,CLUB JOVENTUT BADALONA,90-97,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,90,97,79,97
21,3,HOZONO GLOBAL JAIRIS,MOVISTAR ESTUDIANTES,69-58,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,69,58,69,43
27,4,MOVISTAR ESTUDIANTES,CADI LA SEU,97-83,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,97,83,87,83
39,5,VALENCIA B.C.,MOVISTAR ESTUDIANTES,84-61,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,84,61,84,53
44,6,IDK EUSKOTREN,MOVISTAR ESTUDIANTES,82-70,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,82,70,82,56
50,7,MOVISTAR ESTUDIANTES,SPAR GIRONA,79-88,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,79,88,68,88
58,8,CASADEMONT ZARAGOZA,MOVISTAR ESTUDIANTES,72-66,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,72,66,72,54
67,9,MOVISTAR ESTUDIANTES,BAXI FERROL,83-86,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,83,86,76,86
74,10,INNOVA-TSN LEGANES,MOVISTAR ESTUDIANTES,68-61,https://baloncestoenvivo.feb.es/Partido.aspx?p...,True,68,61,68,51


In [74]:
wrong_2526 = check_games(df_games_2526, df_box_2526)

# Which team appears in all the wrong games?
teams_in_wrong = pd.concat([wrong_2526["home"], wrong_2526["away"]])
print(teams_in_wrong.value_counts().head(3))

wrong_2526[["round", "home", "away", "result", "home_box", "away_box"]].head(10)

240 games checked, 30 do not match
MOVISTAR ESTUDIANTES       30
LOINTEK GERNIKA BIZKAIA     2
HOZONO GLOBAL JAIRIS        2
Name: count, dtype: int64


,round,home,away,result,home_box,away_box
1,1,LOINTEK GERNIKA BIZKAIA,MOVISTAR ESTUDIANTES,89-88,89,68
12,2,MOVISTAR ESTUDIANTES,CLUB JOVENTUT BADALONA,90-97,79,97
21,3,HOZONO GLOBAL JAIRIS,MOVISTAR ESTUDIANTES,69-58,69,43
27,4,MOVISTAR ESTUDIANTES,CADI LA SEU,97-83,87,83
39,5,VALENCIA B.C.,MOVISTAR ESTUDIANTES,84-61,84,53
44,6,IDK EUSKOTREN,MOVISTAR ESTUDIANTES,82-70,82,56
50,7,MOVISTAR ESTUDIANTES,SPAR GIRONA,79-88,68,88
58,8,CASADEMONT ZARAGOZA,MOVISTAR ESTUDIANTES,72-66,72,54
67,9,MOVISTAR ESTUDIANTES,BAXI FERROL,83-86,76,86
74,10,INNOVA-TSN LEGANES,MOVISTAR ESTUDIANTES,68-61,68,51


In [75]:
g = df_games_2526.loc[1]                         # round 1: Lointek Gernika 89-88 Movistar Estudiantes
r = requests.get(g["game_url"], headers=headers)

t_text  = pd.read_html(StringIO(r.text))[1]                         # Estudiantes table (away = table 1)
t_links = pd.read_html(StringIO(r.text), extract_links="body")[1]

check_names = pd.DataFrame({
    "player": t_text.iloc[:, 2],                         # name as text
    "url":    [cell[1] for cell in t_links.iloc[:, 2]],  # link of the name (None if there is no link)
    "pts":    t_text.iloc[:, 4],
})
check_names

,player,url,pts
0,"BUHNER, FRIEDA MIZZI",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,15
1,"STANACEV, ALEKSANDRA",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,5
2,"SAMSON, CLEMENTINE",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,4
3,"ETXARRI MUNARRIZ, IRATI",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,10
4,"KRIVACEVIC, TIJANA",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,0
5,"TOPUZOVIC, KRISTINA",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,18
6,"CAMILION CUCCARO, JUANA",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,10
7,"ODOM, LEAONNA LANEAH",https://baloncestoenvivo.feb.es/Jugador.aspx?i...,6
8,NaN,NaN,68


In [76]:
wrong_2526["missing_pts"] = (
    (wrong_2526["home_pts"] - wrong_2526["home_box"]) + (wrong_2526["away_pts"] - wrong_2526["away_box"])
)
print("Points missing in total:", wrong_2526["missing_pts"].sum())
print("Per game (Estudiantes):", round(wrong_2526["missing_pts"].mean(), 1))

Points missing in total: 372
Per game (Estudiantes): 12.4


**Known data gap (2025/26):** in all 30 games of Movistar Estudiantes, the official FEB box score adds up to
fewer points than the official result (e.g. 68 vs 88 in round 1; the FEB's own totals row says 68).
In total 372 points are missing, 12.4 per game: at least one regular player is missing from every box score of that team.
This is a gap in the source, not in our parsing: every other game (210 of 240, and all 56 in 2026/27) matches exactly.
Consequence: that player is not in the 2025/26 sample. Team possessions are taken from the official team table,
not from the sum of the box scores, so the other Estudiantes players are not affected.

In [77]:
def season_totals(df_box):
    """From one row per player per game to one row per player for the season."""
    info = df_box.groupby("player_url").agg(
        player=("player", "first"),
        team=("team", "last"),
        games=("game_url", "nunique"),
        starts=("starter", "sum"),
    ).reset_index()
    totals = df_box.groupby("player_url")[sum_cols].sum().reset_index()
    return pd.merge(info, totals, on="player_url")

df_players_2526 = season_totals(df_box_2526)

print(df_players_2526.shape)
df_players_2526.sort_values("pts", ascending=False).head(10)

(248, 27)


,player_url,player,team,games,starts,min,pts,oreb,dreb,reb,...,val,plus_minus,fg2_made,fg2_att,fg3_made,fg3_att,fg_made,fg_att,ft_made,ft_att
228,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"COULIBALY, MARIAM ALOU",SPAR GIRONA,29,27,732.366667,504,57,152,209,...,628,232.0,166,244,28,62,194,306,88,114
240,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"BUHNER, FRIEDA MIZZI",MOVISTAR ESTUDIANTES,29,28,876.133333,485,45,150,195,...,540,9.0,156,262,31,127,187,389,80,98
196,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"KONE, SIKA",INGENIERÍA AMBIENTAL CAB ESTEPONA,29,29,875.833333,451,84,172,256,...,598,28.0,132,249,27,66,159,315,106,132
169,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"GIOMI, LYDIA MORA",DURÁN MAQUINARIA ENSINO,30,30,891.616667,438,111,132,243,...,490,103.0,178,354,0,5,178,359,82,113
201,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"CONNER, MADISON ELIZABETH",INGENIERÍA AMBIENTAL CAB ESTEPONA,29,28,828.933333,425,9,71,80,...,333,18.0,65,137,83,204,148,341,46,56
15,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"DEMBELE, GNERE ALIMA",CLUB JOVENTUT BADALONA,25,25,784.783333,424,42,146,188,...,423,-67.0,114,238,35,109,149,347,91,115
49,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"MARTIN CARRION, IYANA",PERFUMERIAS AVENIDA,29,26,749.166667,414,29,60,89,...,413,30.0,103,228,42,123,145,351,82,105
215,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"SANGARE, AMINATA BRAHIMA",INNOVA-TSN LEGANES,30,27,855.533333,405,92,239,331,...,545,-4.0,134,263,19,81,153,344,80,109
98,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"JANKOVIC, MASA",LOINTEK GERNIKA BIZKAIA,28,27,745.966667,385,45,86,131,...,426,-32.0,147,244,3,16,150,260,82,118
86,https://baloncestoenvivo.feb.es/Jugador.aspx?i...,"MERCERON, AXELLE CECILE ELISABETH",SPAR GRAN CANARIA,21,20,764.166667,385,59,113,172,...,496,-274.0,132,255,9,34,141,289,94,128


In [78]:
# Team id of last season = the 'i=' number inside the player URLs
df_players_2526["team_id"] = df_players_2526["player_url"].str.extract(r"i=(\d+)")[0]

team_ids_2526 = df_players_2526["team_id"].unique()
print(len(team_ids_2526), "team ids")

# Test: does the team page of last season show last season's roster?
test_id = team_ids_2526[0]
test_roster = get_roster(f"https://baloncestoenvivo.feb.es/Equipo.aspx?i={test_id}", "")
test_roster[["player", "position", "height"]].head()

16 team ids


,player,position,height
0,BRITTANY AUTUMN BREWER,Pívot,196
1,REBECA COTANO DE JUAN,A-Piv,186
2,GNERE ALIMA DEMBELE,A-Pivot,190
3,LAUREN-JENEE NNEKA EBO,Pívot,193
4,LAURA RACHEL HOWARD,Alero,178


In [79]:
# 1. All rosters of last season (16 team pages)
rosters_2526 = []
for tid in team_ids_2526:
    try:
        rosters_2526.append(get_roster(f"https://baloncestoenvivo.feb.es/Equipo.aspx?i={tid}", ""))
    except Exception as e:
        print("ERROR in team", tid, e)
    time.sleep(1)

df_rosters_2526 = pd.concat(rosters_2526, ignore_index=True)
info_2526 = df_rosters_2526[["player_url", "position", "height", "nationality"]].drop_duplicates("player_url")

df_players_2526 = pd.merge(df_players_2526, info_2526, on="player_url", how="left")
df_players_2526["height"] = pd.to_numeric(df_players_2526["height"], errors="coerce")
print("2025/26 without height:", df_players_2526["height"].isna().sum())

2025/26 without height: 78


In [80]:
# The player id ('c=' in the URL) is the same in every season and every team
for df in [df_players_2526, df_players_2627]:
    df["player_id"] = df["player_url"].str.extract(r"c=(\d+)")[0]

def heights_by_id(df):
    """One known height per player id."""
    return df.dropna(subset=["height"]).drop_duplicates("player_id").set_index("player_id")["height"]

# Fill each season's gaps with the other season's height
df_players_2526["height"] = df_players_2526["height"].fillna(df_players_2526["player_id"].map(heights_by_id(df_players_2627)))
df_players_2627["height"] = df_players_2627["height"].fillna(df_players_2627["player_id"].map(heights_by_id(df_players_2526)))

# Who is still missing, among players with enough minutes (paper: 300 min in a full season)?
print("2025/26, >= 300 min, no height:")
print(df_players_2526[(df_players_2526["min"] >= 300) & df_players_2526["height"].isna()][["player", "team", "min"]])
print("\n2026/27, >= 70 min, no height:")
print(df_players_2627[(df_players_2627["min"] >= 70) & df_players_2627["height"].isna()][["player", "team", "min"]])

2025/26, >= 300 min, no height:
                                player                               team  \
63            CONTELL SANCHIS, CLAUDIA                  SPAR GRAN CANARIA   
78               GEORGE, ANISHA BRIANA                  SPAR GRAN CANARIA   
86   MERCERON, AXELLE CECILE ELISABETH                  SPAR GRAN CANARIA   
152           CARRERA QUINTANA, RAQUEL                      VALENCIA B.C.   
203          GARRICK, MADELEINE CLAIRE  INGENIERÍA AMBIENTAL CAB ESTEPONA   

            min  
63   330.966667  
78   448.766667  
86   764.166667  
152  437.050000  
203  403.000000  

2026/27, >= 70 min, no height:
                      player                      team         min
22        SONTAG, LINA MERLE               CADI LA SEU  123.516667
42   MCSPADDEN, LARA GWENETH  INVERSUS GERNIKA BIZKAIA  113.400000
58   SUAREZ RODRIGUEZ, MARTA             VALENCIA B.C.   92.683333
65   HERNANDEZ LOPEZ, ISABEL             VALENCIA B.C.   76.316667
153     YURKEVICHUS, TETIANA  

In [81]:
df_players_2526.to_csv("data/season_2025_26.csv", index=False)
df_players_2627.to_csv("data/season_2026_27.csv", index=False)
df_games_2526.to_csv("data/games_2025_26.csv", index=False)
df_games.to_csv("data/games_2026_27.csv", index=False)
print("Saved")

Saved


## 1.16 Team totals for 2025/26
Possessions must come from the official team table. We reuse the form trick from 1.14,
now as a function that works for any page of the site.

In [83]:
def get_season_page(url, season):
    """Return the HTML of any page of the FEB site for one season (e.g. 2025 for 2025/26)."""
    session = requests.Session()
    first = session.get(url, headers=headers)

    form = lxml.html.fromstring(first.text).forms[0]
    fields = dict(form.form_values())

    # Find the season dropdown by its content (its name contains 'temporadas'), not by copying it by hand
    season_field = [name for name in fields if "temporadas" in name][0]

    fields[season_field] = str(season)
    fields["__EVENTTARGET"] = season_field.replace(":", "$")   # same name, with $ instead of :
    fields["__EVENTARGUMENT"] = ""

    second = session.post(url, data=fields, headers=headers)
    return second.text

In [84]:
league_url = "https://baloncestoenvivo.feb.es/estadisticas.aspx?g=4&t=2026&nm=lfendesa"
html_teams_2526 = get_season_page(league_url, 2025)

pos = html_teams_2526.find('class="temporada"')
print(html_teams_2526[pos : pos + 50])           # must say 2025/2026

df_teams_2526 = pd.read_html(StringIO(html_teams_2526))[0]
print(df_teams_2526.shape)                        # expected: (16, 20)

class="temporada">2025/2026</span>
        </div>
(8, 20)


In [85]:
def clean_team_table(df):
    """League team table (text, two header levels) -> one clean row per team, with possessions."""
    df = df.copy()
    df.columns = clean_columns
    for col in count_cols:
        df[col] = df[col].apply(get_total)
    for col in shot_cols:
        df[col + "_made"] = df[col].apply(lambda c: get_made_att(c)[0])
        df[col + "_att"]  = df[col].apply(lambda c: get_made_att(c)[1])
    df = df.drop(columns=shot_cols)
    df["poss"] = df["fg_att"] - df["oreb"] + df["tov"] + 0.44 * df["ft_att"]
    df["poss_per_game"] = df["poss"] / df["games"]
    return df

df_teams_2526 = clean_team_table(df_teams_2526)
df_teams_2526[["team", "games", "pts", "poss_per_game"]].sort_values("poss_per_game", ascending=False)

,team,games,pts,poss_per_game
6,INNOVA-TSN LEGANES,2,145,77.66
4,DURÁN MAQUINARIA ENSINO,2,142,75.82
1,CASADEMONT ZARAGOZA,6,448,73.66
3,VALENCIA B.C.,6,447,72.56
7,SPAR GIRONA,4,310,72.31
5,INGENIERÍA AMBIENTAL CAB ESTEPONA,2,117,70.90
0,PERFUMERIAS AVENIDA,4,271,68.87
2,HOZONO GLOBAL JAIRIS,2,120,67.64


In [86]:
pts_box_2526 = df_box_2526.groupby("team")["pts"].sum()
check_2526 = pd.DataFrame({"box_scores": pts_box_2526, "team": df_teams_2526.set_index("team")["pts"]})
check_2526["difference"] = check_2526["team"] - check_2526["box_scores"]
check_2526.sort_values("difference", ascending=False).head(3)

,box_scores,team,difference
team,,,
PERFUMERIAS AVENIDA,2083,271.0,-1812.0
VALENCIA B.C.,2286,447.0,-1839.0
INNOVA-TSN LEGANES,2065,145.0,-1920.0


In [87]:
df_teams_2526["season"] = 2025
df_teams["season"] = 2026
df_teams_2526.to_csv("data/teams_2025_26.csv", index=False)
df_teams.to_csv("data/teams_2026_27.csv", index=False)

In [88]:
form_2526 = lxml.html.fromstring(html_teams_2526).forms[0]

for sel in form_2526.xpath(".//select"):                 # every dropdown list in the form
    print(sel.name)
    for opt in sel.xpath("./option"):                    # every option of that list
        selected = "  <- selected" if opt.get("selected") else ""
        print("    ", opt.get("value"), "|", opt.text_content().strip(), selected)

_ctl0:MainContentPlaceHolderMaster:temporadasDropDownList
     2026 | 2026/2027 
     2025 | 2025/2026   <- selected
     2024 | 2024/2025 
     2023 | 2023/2024 
     2022 | 2022/2023 
     2021 | 2021/2022 
     2020 | 2020/2021 
     2019 | 2019/2020 
     2018 | 2018/2019 
     2017 | 2017/2018 
     2016 | 2016/2017 
     2015 | 2015/2016 
     2014 | 2014/2015 
     2013 | 2013/2014 
     2012 | 2012/2013 
     2011 | 2011/2012 
     2010 | 2010/2011 
     2009 | 2009/2010 
     2008 | 2008/2009 
     2007 | 2007/2008 
     2006 | 2006/2007 
     2005 | 2005/2006 
     2004 | 2004/2005 
     2003 | 2003/2004 
     2002 | 2002/2003 
     2001 | 2001/2002 
     2000 | 2000/2001 
     1999 | 1999/2000 
     1998 | 1998/1999 
     1997 | 1997/1998 
_ctl0:MainContentPlaceHolderMaster:fasesGruposDropDownList
     -44443 | Play-Offs   <- selected
     88860 | Play-Offs Final 
     88859 | Play-Offs 1/2 Final 
     88858 | Play-Offs 1/4 Final 
     88870 | Liga Regular Único 
     -44442

**Check failed:** for 2025/26 the page returned the **playoff** table (8 teams, 2 to 6 games), because the site shows
the last phase of a past season by default. The validation caught it: differences of about −1,800 points per team.
Fix below: choose the regular-season phase as well as the season.

### Fix: choose the phase too
The page has two dropdowns: season and phase. For a past season it shows the playoffs by default,
so we send the form twice: first the season, then the regular-season phase (found by its text, not its number).

In [89]:
def post_dropdown(session, url, html, name_part, value):
    """Send the page's form back with one dropdown changed. Returns the new page HTML."""
    form = lxml.html.fromstring(html).forms[0]
    fields = dict(form.form_values())
    field = [name for name in fields if name_part in name][0]   # the dropdown, found by part of its name
    fields[field] = value
    fields["__EVENTTARGET"] = field.replace(":", "$")
    fields["__EVENTARGUMENT"] = ""
    return session.post(url, data=fields, headers=headers).text


def get_season_page(url, season, phase=None):
    """HTML of any FEB page for one season and, optionally, one phase (e.g. phase='Liga Regular')."""
    session = requests.Session()
    html = session.get(url, headers=headers).text
    html = post_dropdown(session, url, html, "temporadas", str(season))        # 1. choose the season

    if phase is not None:                                                     # 2. choose the phase
        form = lxml.html.fromstring(html).forms[0]
        phase_list = [s for s in form.xpath(".//select") if "fases" in s.name][0]
        value = [o.get("value") for o in phase_list.xpath("./option") if phase in o.text_content()][0]
        html = post_dropdown(session, url, html, "fases", value)
    return html

In [96]:
html_teams_2526 = get_season_page(league_url, 2025, phase="Liga Regular")

df_teams_2526_raw = pd.read_html(StringIO(html_teams_2526))[0]   # raw: as it comes from the site
print(df_teams_2526_raw.shape)                                    # expected: (16, 20)

(16, 20)


In [97]:
df_teams_2526 = clean_team_table(df_teams_2526_raw)   # clean version, new name: safe to run again

pts_box_2526 = df_box_2526.groupby("team")["pts"].sum()
check_2526 = pd.DataFrame({"box_scores": pts_box_2526, "team": df_teams_2526.set_index("team")["pts"]})
check_2526["difference"] = check_2526["team"] - check_2526["box_scores"]
check_2526.sort_values("difference", ascending=False).head(3)

,box_scores,team,difference
team,,,
MOVISTAR ESTUDIANTES,1903,2275,372
BAXI FERROL,2136,2136,0
CADI LA SEU,2037,2037,0


**Result:** with the regular-season phase, the official team points match the box scores for 15 of 16 teams.
The only difference is Movistar Estudiantes, 372 points: the same gap found game by game in 1.15, now confirmed by a second, independent source.

In [98]:
df_teams_2526["season"] = 2025
df_teams["season"] = 2026
df_teams_2526.to_csv("data/teams_2025_26.csv", index=False)
df_teams.to_csv("data/teams_2026_27.csv", index=False)
print("Saved")

Saved


## Summary of the data collection
- **Source:** official FEB stats site (baloncestoenvivo.feb.es), Liga Femenina Endesa, regular season only.
- **Method:** game by game (calendar → box score of every game → season totals per player), because the roster pages only list current players and missed 103 points at Valencia.
- **2025/26:** 240 games, 4,659 player-game rows, 248 players. **2026/27 (rounds 1-7):** 56 games, 1,124 player-game rows, 192 players.
- **Checks against official results:** every game matches its official score, except the 30 games of Movistar Estudiantes in 2025/26, where the FEB box scores miss 372 points (12.4 per game). The official team table confirms the same 372-point gap, so it is a gap in the source, not in our parsing.
- **Team tables:** collected for both seasons, regular season only (a past season shows the playoffs by default, so the phase has to be chosen too). Team possessions come from these tables, not from the box scores.
- **Height and position:** from the team rosters, joined by player URL. Missing heights were filled with the other season (same player id, `c=` in the URL). 10 players with enough minutes still have no height: filled by hand in the next notebook, with the source recorded.
- **Files in `data/`:** `boxscores_2025_26.csv`, `boxscores_2026_27.csv`, `season_2025_26.csv`, `season_2026_27.csv`, `games_2025_26.csv`, `games_2026_27.csv`, `teams_2025_26.csv`, `teams_2026_27.csv`.